### Paper Analysis Notebook - Full Simulation

This notebook runs a full simulation of the modelling framework. It runs n (user defined, but typically 10,000) years of simulations for all possible scenarios. The modelling framework runs 1. flood risk model -> 2. macro economic model -> 3. credit rating model. To speed up simulations, the macromodel has been pre-simulated and this Monte Carlo framework samples from the parameter space of the pre-simulation. The credit rating model is set-up within this notebook. The three individual models in this modelling framework can also be executed in isolation in the notebooks in "individual_models" folder.

The total runtime for the full modelling framework is 3 hours (assuming 10,000 simulations)

Prerequisites: 

    - All notebooks in the "preperation" and "pre_sim" folders should have been executed

Outputs: 
    
    - dataframe for all simulation years and scenarios saved in CSV format

In [ ]:
from pathlib import Path
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import country_converter as coco
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from matplotlib.colors import ListedColormap, BoundaryNorm
from sovereign.flood import build_basin_curves, BasinLossCurve, risk_data_future_shift, run_simulation, extract_sectoral_losses
from itertools import product

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

##### 1. User config

In [ ]:
n_years = 10000 # number of years to simulate
# Flood modelling parameters
adaptation_aep = 0.01 # 100-year flood protection
hydro_model = 'jules-w2' # what hydrological model are we using
EPOCHS = [2030, 2040, 2050, 2060, 2070] # what future epochs are we interested in?
SCENARIOS = ['ssp126', 'ssp370', 'ssp585'] # what climate scenarios are we intersted in?
STATS = ['q05', 'q50', 'q95', 'mean'] # what future climate model stats are we intersted in ?
# Macro modelling parameters (make sure these are the same as "3_dignad_per_simulation.ipynb")
Thai_GDP = 496e9 # 2022 numbers in USD
# National GVA figures from DOSE
agr_GVA = 42880325599
man_GVA = 162659433019
ser_GVA = 316647778583
# Disaggregate output losses
TRADABLE_SHARES = {
    "Agriculture": 1.0,
    "Manufacturing": 0.7,
    "Service": 0.5,
}

##### 2. Set filepaths, load and prepare data

In [ ]:
root = Path.cwd().parent.parent # find project root
risk_data = pd.read_csv(os.path.join(root, 'outputs', 'flood', 'risk', 'basins', 'risk_basins.csv')) # basin-scale flood risk info
copula_random_numbers = pd.read_parquet(os.path.join(root, 'outputs', 'flood', 'dependence', 'copulas', 'copula_random_numbers.gzip')) # copula-derived random #s capturing flood correlation
future_rp_shifts = pd.read_csv(os.path.join(root, 'outputs', 'flood', 'future', 'basin_rp_shifts.csv')) # isimip derived flood frequency changes
macro_presim_noadapt = pd.read_csv(os.path.join(root, 'outputs', 'macro', 'DIGNAD_presim_n1000_noadapt.csv')) # macro presim (no adaptation)
macro_presim_adapt = pd.read_csv(os.path.join(root, 'outputs', 'macro', 'DIGNAD_presim_n1000_adapt.csv')) # macro presim (adaptation)
# Credit rating model calibration data
economic = pd.read_csv(os.path.join(root, 'inputs', 'credit_risk', 'economic.csv')) # macroeconomic and fiscal data
SP_data = pd.read_csv(os.path.join(root, 'inputs', 'credit_risk', 'T3.csv')) # S&P losses and rating data
PD_data = pd.read_csv(os.path.join(root, 'inputs', 'credit_risk', 'PD_ratings.csv'), header=None) # numeric rating and PD relationship
# Final output path
final_output = os.path.join(root, 'outputs', 'results', 'full_model_simulation.csv')
# Clean up risk data
risk_data = risk_data.iloc[:, 1:] # Drop first "unnamed column"
risk_data['AEP'] = 1 / risk_data['RP'] # Add AEP column
risk_data['Pr_L_AEP'] = np.where(risk_data['Pr_L'] == 0, 0, 1 / risk_data['Pr_L']) # Add a column converting current prorection level into AEP
risk_data.reset_index(drop=True, inplace=True) 

##### 3. Prepare and calibrate credit rating model

In [ ]:
gdp_losses = SP_data["GDP_per_capita"] / 100

def polyfit_raw(x, y):
    coeffs = np.polyfit(x, y, 3)
    return coeffs[::-1]   # reverse to match β0 + β1x + β2x² + β3x³

# NGGD
NGGD = np.log(SP_data["NGGD"])
b_NGGD = polyfit_raw(gdp_losses, NGGD)

# GGB (filter < 0)
sub = SP_data[SP_data["GGB"] < 0]
GGB = np.log(-sub["GGB"])
b_GGB = polyfit_raw(sub["GDP_per_capita"]/100, GGB)

# NNED (filter > 0)
sub = SP_data[SP_data["NNED"] > 0]
NNED = np.log(sub["NNED"])
b_NNED = polyfit_raw(sub["GDP_per_capita"]/100, NNED)

# CAB
CAB = np.log(-SP_data["CAB"])
b_CAB = polyfit_raw(gdp_losses, CAB)

# =========================================================
# Helper: apply polynomial
# =========================================================

def equa(A, b0, b1, b2, b3):
    return b0 + b1*A + b2*A**2 + b3*A**3

# Real GDP growth (pct change)
economic["S_RealGDPgrowth"] = economic.groupby("CountryName")["S_GDPpercapitaUS"].pct_change()

economic = economic[
    [
        "CountryName","Year","scale20","S_GDPpercapitaUS",
        "S_RealGDPgrowth","S_NetGGdebtGDP","S_GGbalanceGDP",
        "S_NarrownetextdebtCARs","S_CurrentaccountbalanceGDP"
    ]
]

cc = coco.CountryConverter()
economic["ISO2"] = cc.convert(economic["CountryName"], to="ISO3")

# Build baseline data
Baseline = economic.copy()
Baseline["ln_S_GDPpercapitaUS"] = np.log(Baseline["S_GDPpercapitaUS"])

Baseline = Baseline[Baseline["Year"] > 2014]
Baseline = Baseline.dropna()

rating = PD_data.iloc[:, 0]
default = PD_data.iloc[:, 1]
b_PD = polyfit_raw(rating, default)

def implement_PD_equation(r):
    PD = equa(r, *b_PD)
    return np.clip(PD, 0, 100)

def calculate_spreads(C_0, notches):
    C_0 = np.asarray(C_0)
    notches = np.asarray(notches)
    return (-282.51 * notches) + (14.23 * notches * C_0)

def estimate_country_scenario(country_, loss_):

    # extract baseline for 2020
    gdp_per_capita = Baseline[(Baseline["CountryName"] == country_) & (Baseline["Year"] == 2020)].iloc[0]

    gdp_pc_value = gdp_per_capita["S_GDPpercapitaUS"]

    estimation = pd.DataFrame({
        "CountryName": [country_],
        "loss": [loss_]
    })

    estimation["ISO2"] = cc.convert(estimation["CountryName"], to="ISO3")
    estimation["ln_S_GDPpercapitaUS"] = np.log(gdp_pc_value * (1 - loss_))
    estimation["S_RealGDPgrowth"] = -loss_

    # baseline values
    for col in ["S_NetGGdebtGDP","S_GGbalanceGDP","S_NarrownetextdebtCARs","S_CurrentaccountbalanceGDP"]:
        estimation[col] = gdp_per_capita[col]

    A = -loss_

    # apply fitted polynomial adjustments
    estimation["S_NetGGdebtGDP"] += np.exp(equa(A, *b_NGGD))
    estimation["S_GGbalanceGDP"] += -np.exp(equa(A, *b_GGB))
    estimation["S_NarrownetextdebtCARs"] += np.exp(equa(A, *b_NNED))
    estimation["S_CurrentaccountbalanceGDP"] += -np.exp(equa(A, *b_CAB))

    return estimation

features = [
    "ln_S_GDPpercapitaUS",
    "S_RealGDPgrowth",
    "S_NetGGdebtGDP",
    "S_GGbalanceGDP",
    "S_NarrownetextdebtCARs",
    "S_CurrentaccountbalanceGDP"
]

X_train = Baseline[features]
y_train = Baseline["scale20"]

rf = RandomForestRegressor(
    n_estimators=2000,
    random_state=77,
    oob_score=True
)
rf.fit(X_train, y_train)

##### 4. Prepare basin loss curves

In [ ]:
# Adjust baseline risk to future risk 
future_risk = {}  # (scenario, epoch, stat) -> adjusted risk_data
for scenario, epoch, stat in product(SCENARIOS, EPOCHS, STATS):
    future_risk[(scenario, epoch, stat)] = risk_data_future_shift(
        risk_data,
        future_rp_shifts,
        hydro_model,
        scenario,
        epoch,
        stat,
        degrade_protection=True
    )
# Build baseline curves
baseline_curves: dict[int, BasinLossCurve] = build_basin_curves(risk_data)
future_curves = {}
# Build future curves
for key, future_risk_data in future_risk.items():
    future_curves[key] = build_basin_curves(future_risk_data)

##### 5. Prepare macro interpolators

In [ ]:
# DIGNAD INTERPOLATOR
# notation: _na (no adaptation) _a (adaptation)
X_na = macro_presim_noadapt[['dY_T', 'dY_N', 'dK_priv', 'dK_pub']].values
y_na = macro_presim_noadapt['gdp_avg'].values
X_a = macro_presim_adapt[['dY_T', 'dY_N', 'dK_priv', 'dK_pub']].values
y_a = macro_presim_adapt['gdp_avg'].values

linear_interp_na = LinearNDInterpolator(X_na, y_na)
nearest_interp_na = NearestNDInterpolator(X_na, y_na)
linear_interp_a = LinearNDInterpolator(X_a, y_a)
nearest_interp_a = NearestNDInterpolator(X_a, y_a)

def _as_scalar(x):
    arr = np.asarray(x)
    if arr.size != 1:
        raise ValueError(f"Expected scalar interpolation result, got shape {arr.shape}")
    return float(arr.item())

def interpolate_gdp(params, adaptation=False):
    if adaptation:
        gdp = linear_interp_a(params)
        if np.isnan(_as_scalar(gdp)):
            gdp = nearest_interp_a(params)
    else:
        gdp = linear_interp_na(params)
        if np.isnan(_as_scalar(gdp)):
            gdp = nearest_interp_na(params)

    return _as_scalar(gdp)

##### 6. Full simulation

In [ ]:
# First define the simulation functions
def flood_aggregates_one_year(basin_curves, random_ns_row, adaptation=False, adaptation_aep=0.01):
    """
    Returns annual monetary losses aggregated across basins for the 5 sectors needed downstream.
    """
    ag = man = serv = priv = pub = 0.0

    for basin_id, curve in basin_curves.items():
        basin_str = str(int(basin_id))
        if basin_str not in random_ns_row:
            continue

        aep_event = 1 - random_ns_row[basin_str]

        if adaptation:
            ag   += curve.loss_at_event_aep(aep_event, scenario="adaptation", adapted_protection_aep=adaptation_aep, sector="Agriculture")
            man  += curve.loss_at_event_aep(aep_event, scenario="adaptation", adapted_protection_aep=adaptation_aep, sector="Manufacturing")
            serv += curve.loss_at_event_aep(aep_event, scenario="adaptation", adapted_protection_aep=adaptation_aep, sector="Service")
            priv += curve.loss_at_event_aep(aep_event, scenario="adaptation", adapted_protection_aep=adaptation_aep, sector="Private")
            pub  += curve.loss_at_event_aep(aep_event, scenario="adaptation", adapted_protection_aep=adaptation_aep, sector="Public")    
        else:
            ag   += curve.loss_at_event_aep(aep_event, scenario="baseline", sector="Agriculture")
            man  += curve.loss_at_event_aep(aep_event, scenario="baseline", sector="Manufacturing")
            serv += curve.loss_at_event_aep(aep_event, scenario="baseline", sector="Service")
            priv += curve.loss_at_event_aep(aep_event, scenario="baseline", sector="Private")
            pub  += curve.loss_at_event_aep(aep_event, scenario="baseline", sector="Public")

    return ag, man, serv, priv, pub

def run_integrated_yearwise(
    country_: str,
    baseline_curves: dict,
    future_curves: dict,              # (scenario, epoch, stat) -> basin_curves
    copula_random_ns: pd.DataFrame,
    n_years: int,
    adaptation_aep: float,
    # mapping constants
    agr_GVA: float,
    man_GVA: float,
    ser_GVA: float,
    tradable_shares: dict,
    thai_gdp: float,
):
    # Combine curve sets: baseline + futures
    curve_sets = {"baseline": baseline_curves, **future_curves}

    # Precompute denominators for macro mapping
    tradable_output_baseline = (
        agr_GVA * tradable_shares["Agriculture"] +
        man_GVA * tradable_shares["Manufacturing"] +
        ser_GVA * tradable_shares["Service"]
    )
    nontrad_output_baseline = (
        agr_GVA * (1 - tradable_shares["Agriculture"]) +
        man_GVA * (1 - tradable_shares["Manufacturing"]) +
        ser_GVA * (1 - tradable_shares["Service"])
    )

    # Pull baseline rating once (like your function does)
    original_rating = float(
        Baseline.loc[
            (Baseline["CountryName"] == country_) &
            (Baseline["Year"] == 2020),
            "scale20"
        ].iloc[0]
    )
    original_pd = implement_PD_equation(original_rating)

    rows = []

    adaptation_scenarios = [True, False]

    # For batch RF prediction for credit rating model (otherwise runtime is several days)
    X_rows = []          # will become DataFrame for rf.predict
    meta_idx = []        # track alignment: one entry per X_row

    for t in tqdm(range(n_years), desc="Simulating years"):
        random_ns_row = copula_random_ns.loc[t]

        for adaptation in adaptation_scenarios:
            for key, basin_curves in curve_sets.items():
                # --- 1) FLOOD ---
                if adaptation:
                    ag, man, serv, priv, pub = flood_aggregates_one_year(basin_curves, random_ns_row, adaptation=True, adaptation_aep=adaptation_aep)
                else:
                    ag, man, serv, priv, pub = flood_aggregates_one_year(basin_curves, random_ns_row, adaptation=False)
                    
                # --- 2) MAP TO MACRO PARAMS 
                trad_out = (
                    ag  * tradable_shares["Agriculture"] +
                    man * tradable_shares["Manufacturing"] +
                    serv* tradable_shares["Service"]
                )
                nontrad_out = (
                    ag  * (1 - tradable_shares["Agriculture"]) +
                    man * (1 - tradable_shares["Manufacturing"]) +
                    serv* (1 - tradable_shares["Service"])
                )
    
                dY_T   = trad_out / tradable_output_baseline
                dY_N   = nontrad_out / nontrad_output_baseline
                dK_priv= priv / thai_gdp
                dK_pub = pub  / thai_gdp
    
                # Params vector needs to align with how linear_interp() was trained.
                params = np.array([dY_T, dY_N, dK_priv, dK_pub], dtype=float)
    
                # --- 3) MACRO (GDP loss) ---
                if adaptation:
                    gdp_loss = float(interpolate_gdp(params, adaptation=True))
                else:
                    gdp_loss = float(interpolate_gdp(params, adaptation=False))
    
                # --- 4) CREDIT Ratings (don't predict in loop, batch predict later) ---
                # Build the feature row for RF model and store it
                estimation = estimate_country_scenario(country_, (gdp_loss * -1 / 100))

                # --- Store RF inputs (features) for batch prediction ---
                if isinstance(estimation, pd.Series):
                    est_row = estimation.to_frame().T
                else:
                    est_row = estimation

                X_rows.append(est_row[features].iloc[0].to_dict())  # store as dict 
                meta_idx.append(len(rows))  # this X row corresponds to the row we append next
                
                # --- store ---
                if key == "baseline":
                    scenario = "baseline"
                    epoch = stat = None
                else:
                    scenario, epoch, stat = key  # key is (ssp, epoch, stat)
    
                rows.append({
                    "year_index": t,
                    "scenario": scenario,
                    "epoch": epoch,
                    "stat": stat,
                    "adaptation": adaptation,
                    
                    # flood outputs
                    "AGR_loss": ag,
                    "MAN_loss": man,
                    "SER_loss": serv,
                    "PRI_dam": priv,
                    "PUB_dam": pub,
                    
                    # macro shocks
                    "dY_T": dY_T,
                    "dY_N": dY_N,
                    "dK_priv": dK_priv,
                    "dK_pub": dK_pub,
                    "gdp_loss": gdp_loss,
                    
                    # credit outputs
                    "original_rating": original_rating,
                    "original_pd": original_pd,

                    # placeholders; fill after batch prediction
                    "predicted_rating": np.nan,
                    "predicted_pd": np.nan,
                    "spread_delta": np.nan,
                })

    # Batch Credit Rating RF prediction (significantly speeds up runtimes)
    X = pd.DataFrame(X_rows)

    # Predict ratings in one go
    pred_ratings = rf.predict(X[features]).astype(float)

    # Fill back into rows (keeping row structure unchanged)
    for pr, row_i in zip(pred_ratings, meta_idx):
        rows[row_i]["predicted_rating"] = float(pr)
        rows[row_i]["predicted_pd"] = float(implement_PD_equation(pr))
        rows[row_i]["spread_delta"] = float(calculate_spreads(original_rating, (pr - original_rating)))

    return pd.DataFrame(rows)

In [ ]:
# Simulation run
full_sim = run_integrated_yearwise(
    country_="Thailand",
    baseline_curves=baseline_curves,
    future_curves=future_curves,
    copula_random_ns=copula_random_numbers,
    n_years=n_years,
    adaptation_aep=adaptation_aep,
    agr_GVA=agr_GVA,
    man_GVA=man_GVA,
    ser_GVA=ser_GVA,
    tradable_shares=TRADABLE_SHARES,
    thai_gdp=Thai_GDP
)
# Save to results
Path(final_output).parent.mkdir(parents=True, exist_ok=True) # create directory if it doesn't already exist
full_sim.to_csv(final_output)